<a href="https://colab.research.google.com/github/Naveen-gale/deep_learning/blob/main/ppt_promte_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("ppt_prompt_dataset.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'ppt_prompt_dataset.csv'

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df["input"] = (
    "Topic: " + df["topic"] +
    "\nAudience: " + df["audience"] +
    "\nDifficulty: " + df["difficulty"] +
    "\nSlides: " + df["slides"].astype(str) +
    "\nTheme: " + df["theme"] +
    "\nLanguage: " + df["language"] +
    "\nPresentation Type: " + df["presentation_type"] +
    "\nIndustry: " + df["industry"] +
    "\nNeed Diagrams: " + df["need_diagrams"].astype(str) +
    "\nNeed Images: " + df["need_images"].astype(str) +
    "\nNeed Flowcharts: " + df["need_flowcharts"].astype(str) +
    "\nNeed Tables: " + df["need_tables"].astype(str) +
    "\nNeed Icons: " + df["need_icons"].astype(str)
)

In [ ]:
train_df = df[["instruction","input","output"]]

In [ ]:
train_df

In [ ]:
train_df["text"] = (
    "<|user|>\n"
    + train_df["instruction"]
    + "\n\n"
    + train_df["input"]
    + "\n\n<|assistant|>\n"
    + train_df["output"]
)

In [ ]:
train_df.head()

In [ ]:
train_df[["text"]].to_json(
    "train.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="train.jsonl"
)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct"
)

tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=4096
    )

tokenized_dataset = dataset.map(tokenize)

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct"
)

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


In [ ]:
pip install peft

In [ ]:
!
!pip install "torchao>=0.16.0"
from peft import get_peft_model

model = get_peft_model(model, lora_config)

In [ ]:
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    logging_steps=10,

    save_steps=500,

    save_total_limit=2,

    fp16=True,

    report_to="none"
)

In [ ]:
pip install trl

In [ ]:
from trl import SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model=model,

    train_dataset=tokenized_dataset["train"],

    args=training_args
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("ppt_model")

In [ ]:
tokenizer.save_pretrained("ppt_model")

In [ ]:
print('--- Contents of ppt_model directory after saving ---')
!ls -F ppt_model
print('----------------------------------------------------')

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from peft import PeftModel

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct"
)

In [ ]:
model = PeftModel.from_pretrained(
    base_model,
    "ppt_model"
)

In [ ]:
prompt = """
Create a presentation.

Topic: Machine Learning

Audience: Beginners

Slides: 10

Theme: Modern Blue
"""

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt")

In [ ]:
output = model.generate(
    **inputs,
    max_new_tokens=1500,
    temperature=0.7
)

In [ ]:
print(
    tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )
)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained("ppt_model")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    "ppt_model"
)

The model is now loaded and ready for use. Let's try generating a presentation for a different topic.

In [ ]:
new_prompt = """
Create a presentation.

Topic: Artificial Intelligence in Healthcare

Audience: Medical Professionals

Slides: 7

Theme: Clinical Modern
"""

new_inputs = tokenizer(new_prompt, return_tensors="pt")

new_output = model.generate(
    **new_inputs,
    max_new_tokens=1500,
    temperature=0.7
)

print(
    tokenizer.decode(
        new_output[0],
        skip_special_tokens=True
    )
)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy the saved model to Google Drive
# Remove any existing target directory first to ensure a clean copy
!rm -rf "/content/drive/My Drive/ppt_model_saved_for_hf"
# Verify the existence of ppt_model locally before copying
!echo "--- Current directory contents before copy ---"
!ls -F
!echo "--------------------------------------------"
# Copy the entire 'ppt_model' directory into a new, clearly named directory in Drive
!cp -r ppt_model "/content/drive/My Drive/ppt_model_saved_for_hf"

In [ ]:
!pip install -U huggingface_hub

In [ ]:
from huggingface_hub import login

login()



In [ ]:
from huggingface_hub import create_repo

create_repo(
    repo_id="n99av80n/ppt-prompt-model",
    repo_type="model",
    exist_ok=True
)

In [ ]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path="/content/drive/My Drive/ppt_model_saved_for_hf/ppt_model",
    repo_id="n99av80n/ppt-prompt-model",
    repo_type="model"
)

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files("n99av80n/ppt-prompt-model")
print(files)

In [ ]:
!find "/content/drive/My Drive/ppt_model_saved" -type f

In [ ]:
from huggingface_hub import whoami

print(whoami())

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LORA_MODEL = "n99av80n/ppt-prompt-model" # Corrected repository name casing

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, LORA_MODEL)

print("✅ Model Loaded Successfully")

In [ ]:
prompt = """
Create a professional 10-slide PowerPoint presentation on Machine Learning.
"""

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=800,
    temperature=0.7,
    top_p=0.9
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

In [ ]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path="/content/drive/My Drive/ppt_model_saved",
    repo_id="n99av80n/ppt-prompt-model",
    repo_type="model"
)

In [3]:
from huggingface_hub import list_repo_files

files = list_repo_files("n99av80n/ppt-prompt-model")
print(files)

['.gitattributes', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'ppt_prompt_dataset.csv', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LORA_MODEL = "n99av80n/ppt-prompt-model"

tokenizer = AutoTokenizer.from_pretrained(LORA_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    LORA_MODEL
)

print("✅ Model loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [5]:
!pip uninstall -y torchao
!pip install -U torchao
!pip install -U peft transformers accelerate

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 70.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [1]:
!pip uninstall -y torchao


Found existing installation: torchao 0.17.0
Uninstalling torchao-0.17.0:
  Successfully uninstalled torchao-0.17.0


In [2]:
import torch
import transformers
import peft

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

try:
    import torchao
    print("TorchAO:", torchao.__version__)
except ImportError:
    print("TorchAO: Not installed")

Torch: 2.11.0+cu128
Transformers: 5.14.1
PEFT: 0.19.1
TorchAO: Not installed


In [3]:
!pip uninstall -y transformers peft accelerate torchao

!pip install -U \
  transformers \
  peft \
  accelerate

Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1
Found existing installation: peft 0.17.1
Uninstalling peft-0.17.1:
  Successfully uninstalled peft-0.17.1
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.14.1-py3-none-any.whl (11.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 37.4 MB/s eta 0:00:00
Using cached accelerate-1.14.0-py3-none-any.whl (389 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 57.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2


In [1]:
import transformers
import peft
import huggingface_hub

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("HF Hub:", huggingface_hub.__version__)

Transformers: 4.57.1
PEFT: 0.17.1
HF Hub: 0.36.2


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LORA_MODEL = "n99av80n/ppt-prompt-model"

tokenizer = AutoTokenizer.from_pretrained(LORA_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    LORA_MODEL
)

print("✅ Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


AttributeError: 'list' object has no attribute 'keys'

In [1]:
from huggingface_hub import list_repo_files

print(list_repo_files("n99av80n/ppt-prompt-model"))

['.gitattributes', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'ppt_prompt_dataset.csv', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct"
)

print("Tokenizer loaded!")

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded!


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LORA_MODEL = "n99av80n/ppt-prompt-model"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, LORA_MODEL)

print("✅ Loaded successfully!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 8.68MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

✅ Loaded successfully!


In [4]:
messages = [
    {
        "role": "user",
        "content": "Create a professional 10-slide presentation on Machine Learning."
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=800,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Create a professional 10-slide presentation on Machine Learning.
assistant
Sure! Here's a professional 10-slide presentation on Machine Learning:

Slide 1: Title Slide
- Title of the presentation
- Presenter name and title

Slide 2: Introduction to Machine Learning
- What is Machine Learning?
- Overview of machine learning techniques
- Importance of machine learning in various fields

Slide 3: Types of Machine Learning
- Supervised Learning (Regression, Classification)
- Unsupervised Learning (Clustering, Association)
- Reinforcement Learning (Decision Making)

Slide 4: Supervised Learning
- Example: Logistic Regression
- Steps:
  - Data preparation
  - Model selection
  - Training model
  - Evaluation metrics
  - Predictions

Slide 5: Unsupervised Learning
- Example: K-Means Clustering
- Steps:
  - Data preparation
  - Initialization
  - Iteration
  - Evaluation
  - Decision making

Slide 6: Reinforcement

In [5]:
pip install -U transformers peft accelerate huggingface_hub

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LORA_MODEL = "n99av80n/ppt-prompt-model"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    LORA_MODEL
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]